# 01 - Setup and Validation

Run this first. Validates the reimplemented decision rule against the recorded verdicts (see `graph_utils.py`'s `_self_test` docstring for the diagnosed ~90% match rate, confirmed to be graph-cache drift, not a logic bug), and re-checks Semantic Scholar citation-context coverage at a larger sample size than this session's initial 18-paper probe.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


## Decision-rule validation

In [ ]:
from graph_utils import CitationGraph, load_relational_claims, verify, _self_test

result = _self_test(Path(BENCH).resolve().parents[3] if False else Path('.'))
# _self_test expects a project root with a fixed relative layout; re-run its
# logic directly here against the Kaggle-resolved paths instead:
graph = CitationGraph.load(REFS_CACHE)
claims = load_relational_claims(BENCH, CITE_RESULTS)
with_recorded = [c for c in claims if c['recorded_verdict'] is not None]
matches = sum(1 for c in with_recorded
              if c['paper_a'] and c['paper_b'] and c['relation']
              and verify(c['paper_a'], c['relation'], c['paper_b'], graph) == c['recorded_verdict'])
print(f"graph nodes: {len(graph)}")
print(f"claims with recorded verdict: {len(with_recorded)}")
print(f"match rate: {matches/len(with_recorded):.1%}")
print("Expected ~90% (see graph_utils.py docstring for the graph-drift diagnosis, "
      "not a logic bug -- confirmed via the tau-independent-relation test).")


## Citation-context API coverage (larger sample)

Requires `SEMANTIC_SCHOLAR_API_KEY` as a Kaggle secret for a decent rate limit (unauthenticated requests hit 429 quickly, as this session's initial 18-paper probe did). Confirms whether the text modality for the learned verifier (item 1) can include real citation-context sentences at scale, not just abstract-level similarity.

In [ ]:
import requests, time

try:
    from kaggle_secrets import UserSecretsClient
    S2_KEY = UserSecretsClient().get_secret("SEMANTIC_SCHOLAR_API_KEY")
except Exception:
    S2_KEY = os.environ.get("SEMANTIC_SCHOLAR_API_KEY", "")
HDRS = {"x-api-key": S2_KEY} if S2_KEY else {}
print("S2 key present:", bool(S2_KEY), "(unauthenticated works but rate-limits fast)")

claims = load_relational_claims(BENCH, CITE_RESULTS)
sample_ids = list({c["paper_a"] for c in claims if c["paper_a"]})[:60]

tested, with_context, total_edges, edges_with_context = 0, 0, 0, 0
for pid in sample_ids:
    url = f"https://api.semanticscholar.org/graph/v1/paper/{pid}/citations"
    try:
        r = requests.get(url, params={"fields": "contexts", "limit": 50}, headers=HDRS, timeout=20)
        tested += 1
        if r.status_code != 200:
            time.sleep(1.0)
            continue
        data = r.json().get("data", [])
        has_ctx = False
        for edge in data:
            total_edges += 1
            if edge.get("contexts"):
                edges_with_context += 1
                has_ctx = True
        with_context += int(has_ctx)
    except Exception as e:
        print(pid, "error:", e)
    time.sleep(1.0 if S2_KEY else 3.0)

print(f"papers tested: {tested} | papers with any context: {with_context}")
print(f"edges with context: {edges_with_context}/{total_edges} "
      f"({edges_with_context/max(1,total_edges):.1%})")
